In [22]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
# Imports
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments
from datasets import load_dataset

# Load your dataset
dataset = load_dataset('json', data_files='projects/datasets/article_summary/finnish_articles.json')

Generating train split: 72 examples [00:00, 630.43 examples/s]


In [23]:
print(dataset)


DatasetDict({
    train: Dataset({
        features: ['text', 'summary'],
        num_rows: 72
    })
})


In [24]:

# Load tokenizer and model
model_name = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Preprocessing function
def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["text"],     
        max_length=512,                    
        truncation=True,                  
        padding="max_length"                
    )
    labels = tokenizer(
        examples["summary"],
        max_length=128,                    
        truncation=True,                   
        padding="max_length"               
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

C:\Users\kisme\anaconda3\envs\finalproject\lib\site-packages\transformers\convert_slow_tokenizer.py:559: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [ ]:

# Tokenize
tokenized_dataset = dataset.map(preprocess_function, batched=True)

from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",  # this now works fine
    learning_rate=5e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=30,
    predict_with_generate=True,
    fp16=False,  # True if you have good GPU with mixed precision
)


trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["train"],
    tokenizer=tokenizer,
)

trainer.train()

trainer.save_model("./finetuned-mt5-finnish-summarizer")

Map: 100%|██████████| 72/72 [00:00<00:00, 517.53 examples/s]
C:\Users\kisme\AppData\Local\Temp\ipykernel_19972\1319758973.py:20: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
C:\Users\kisme\anaconda3\envs\finalproject\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss


In [21]:

from datascrape import scrape_article
from transformers import pipeline

summarizer = pipeline("text2text-generation", model="./finetuned-mt5-finnish-summarizer", tokenizer="google/mt5-small")

new_text = scrape_article("https://yle.fi/a/74-20158620", "yle")

input_text = "Yhteenveto: " + new_text

summary = summarizer(input_text, max_length=100, min_length=20, do_sample=False)
print(summary[0]['generated_text'])


C:\Users\kisme\anaconda3\envs\finalproject\lib\site-packages\transformers\convert_slow_tokenizer.py:559: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Device set to use cpu


<extra_id_0> mukaan. – Kaikki ovat kaikki. – Kaikki ovat kaikki.
